In [ ]:

# ─────────────────────────────────────────────────────────────────────────────
#  * User Configuration
# ─────────────────────────────────────────────────────────────────────────────
IMG_DIR     = "/media/desk16/iyun3015/影像组学/processed_data"  # (unchanged, kept as original)
NII_DIR     = "/media/desk16/iyun3015/影像组学"                # (unchanged)
OUT_DIR     = "./fig_all_heatmaps"
PATIENT_ID  = "0039446173"
LABEL_CLASS = None

IMG_SIZE = 224
K        = 3
WIN      = 11

# ─────────────────────────────────────────────────────────────────────────────
import os, glob, warnings, time
warnings.filterwarnings("ignore")

import numpy as np
from scipy.ndimage  import (label as nd_label, binary_fill_holes,
                             zoom, uniform_filter)
from skimage.feature import graycomatrix, local_binary_pattern
from skimage.filters import threshold_otsu, laplace, gaussian as sk_gaussian, sobel
from skimage.morphology import binary_closing, disk, remove_small_objects

try:
    import pywt; HAS_PYWT = True
except ImportError:
    HAS_PYWT = False
    print("  [WARN] PyWavelets not installed, wavelet features will be skipped. "
          "pip install PyWavelets")

try:
    from skimage.filters import gabor as skimage_gabor
    def _gabor_call(img_f32, frequency, theta, sigma_x, sigma_y):
        real, _ = skimage_gabor(img_f32, frequency=frequency, theta=theta,
                                sigma_x=sigma_x, sigma_y=sigma_y)
        return np.abs(real).astype(float)
except TypeError:
    from skimage.filters import gabor_kernel
    from scipy.ndimage import convolve as nd_convolve
    def _gabor_call(img_f32, frequency, theta, sigma_x, sigma_y):
        kernel = np.real(gabor_kernel(frequency, theta=theta,
                                      sigma_x=sigma_x, sigma_y=sigma_y))
        return np.abs(nd_convolve(img_f32, kernel)).astype(float)

from sklearn.cluster       import KMeans
from sklearn.preprocessing import StandardScaler
from PIL import Image

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot   as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors  import LinearSegmentedColormap
from matplotlib.patches import Patch

try:
    import nibabel as nib; HAS_NIB = True
except ImportError:
    HAS_NIB = False

os.makedirs(OUT_DIR, exist_ok=True)

# ══════════════════════════════════════════════════════════════════════════════
#  Color maps
# ══════════════════════════════════════════════════════════════════════════════
BG = "#0d0d1a"

def _cmap(*colors):
    return LinearSegmentedColormap.from_list("", list(colors))

CMAPS = {
    "contrast"   : _cmap("#03045e","#0077b6","#90e0ef","#ffd166","#ef233c"),
    "correlation": _cmap("#0d1321","#1d3461","#2196c3","#b5e8e0","#f0a500"),
    "energy"     : _cmap("#10002b","#3c096c","#b185db","#e0cffc","#f9c74f"),
    "homogeneity": _cmap("#03045e","#48cae4","#90e0ef","#caf0f8","#ade8f4"),
    "entropy"    : _cmap("#1a1a2e","#16213e","#0f3460","#533483","#e94560"),
    "maxprob"    : _cmap("#10002b","#6f3ad0","#e0cffc","#f9c74f","#f3722c"),
    "sre"        : _cmap("#03045e","#0077b6","#00b4d8","#f0f3bd","#ef233c","#9b2226"),
    "lre"        : _cmap("#9b2226","#ef233c","#f0f3bd","#00b4d8","#0077b6","#03045e"),
    "rln"        : _cmap("#1b1b2f","#162447","#1f4068","#1b262c","#0f3460","#533483"),
    "hglre"      : _cmap("#0d1321","#1d3461","#f0a500","#f3722c","#b5e8e0"),
    "gln"        : _cmap("#0d1321","#1d3461","#1f5c92","#6dcff6","#ffeaa7","#c0392b"),
    "sae"        : _cmap("#03045e","#0096c7","#90e0ef","#caf0f8","#ade8f4"),
    "lae"        : _cmap("#ade8f4","#caf0f8","#90e0ef","#0096c7","#03045e"),
    "zp"         : _cmap("#1a1a2e","#16213e","#e94560","#f5a623","#2ecc71"),
    "mean"       : _cmap("#f8f9fa","#dee2e6","#6c757d","#343a40","#0d1117"),
    "std"        : _cmap("#03045e","#0077b6","#48cae4","#ade8f4","#f9c74f","#ef233c"),
    "skewness"   : _cmap("#9b2226","#ef233c","#fefae0","#0077b6","#03045e"),
    "kurtosis"   : _cmap("#1a1a2e","#533483","#f9c74f","#ef233c","#9b2226"),
    "loc_entropy": _cmap("#1a1a2e","#0f3460","#533483","#f9c74f","#ef8c2b"),
    "gabor"      : _cmap("#0d1b2a","#1e6091","#48cae4","#ffd166","#e63946"),
    "lbp"        : _cmap("#1b1b2f","#162447","#1f4068","#1b262c","#e94560"),
    "log"        : _cmap("#03045e","#0096c7","#90e0ef","#ffd166","#ef233c"),
    "gradient"   : _cmap("#0d0d1a","#1d3461","#2196c3","#ffd166","#ef233c","#9b2226"),
    "wavelet"    : _cmap("#10002b","#3c096c","#9b2226","#f9c74f","#2ecc71"),
    "habitat"    : ["#2166AC", "#E8572A", "#1A9641", "#9B2226", "#533483"],
}

LTR_KW = dict(fontsize=15, fontweight="bold", color="white",
              bbox=dict(boxstyle="round,pad=0.2", fc="#111111cc", lw=0))


# ══════════════════════════════════════════════════════════════════════════════
#  Data loading
# ══════════════════════════════════════════════════════════════════════════════
def _auto_patient(img_dir, pid, cls):
    classes = [cls] if cls else ["sensitive", "insensitive"]
    for c in classes:
        d = os.path.join(img_dir, c)
        if not os.path.isdir(d): continue
        for p in sorted(glob.glob(os.path.join(d, "*.png"))):
            fpid = os.path.splitext(os.path.basename(p))[0]
            if pid is None or fpid == pid:
                return fpid, c, p
    raise FileNotFoundError(f"Patient {pid} not found")

def _nii_label_path(nii_dir, pid):
    for pat in [f"{pid}-label.nii.gz", f"{pid}label.nii.gz"]:
        p = os.path.join(nii_dir, pat)
        if os.path.exists(p): return p
    hits = glob.glob(os.path.join(nii_dir, f"{pid}*label*.nii.gz"))
    return hits[0] if hits else None

def _nii_to_mask(nii_path, hw):
    vol = nib.load(nii_path).get_fdata()
    vol = (vol > 0).astype(np.uint8)
    best_ax = max(range(3),
        key=lambda ax: vol.sum(axis=tuple(i for i in range(3) if i != ax)).max())
    area = [vol.take(i, axis=best_ax).sum() for i in range(vol.shape[best_ax])]
    sl   = vol.take(int(np.argmax(area)), axis=best_ax)
    if best_ax != 2: sl = sl.T
    h, w = sl.shape; th, tw = hw
    if (h, w) != (th, tw):
        sl = zoom(sl.astype(float), (th / h, tw / w), order=0)
    return sl > 0.5

def _otsu_mask(gray):
    b = gray > threshold_otsu(gray)
    b = binary_closing(b, disk(6))
    b = binary_fill_holes(b)
    b = remove_small_objects(b, min_size=300)
    lbl, n = nd_label(b)
    if n == 0: return b
    sizes = [(lbl == i).sum() for i in range(1, n + 1)]
    return lbl == (sizes.index(max(sizes)) + 1)

print("=" * 60)
print("  Loading data …")
patient_id, label_class, png_path = _auto_patient(
    IMG_DIR, PATIENT_ID, LABEL_CLASS)
print(f"  Patient ID : {patient_id}  |  Class: {label_class}")

img = np.array(
    Image.open(png_path).convert("L")
         .resize((IMG_SIZE, IMG_SIZE), Image.LANCZOS), dtype=np.uint8)

mask = None
if HAS_NIB:
    nii_p = _nii_label_path(NII_DIR, patient_id)
    if nii_p:
        try:
            mask = _nii_to_mask(nii_p, (IMG_SIZE, IMG_SIZE))
            if mask.sum() < 50:
                print("  NIfTI mask too small → fallback to Otsu"); mask = None
            else:
                print("  Mask source: NIfTI")
        except Exception as e:
            print(f"  NIfTI failed ({e}) → fallback to Otsu"); mask = None
if mask is None:
    mask = _otsu_mask(img)
    print("  Mask source: Otsu")
print(f"  Mask pixels: {mask.sum()}")


# ══════════════════════════════════════════════════════════════════════════════
#  Utility functions
# ══════════════════════════════════════════════════════════════════════════════
def _q(patch, lv=16):
    arr = np.clip(patch, 0, 255).astype(np.uint8)
    return (arr // (256 // lv)).clip(0, lv - 1)

def _norm(feat, mask):
    feat = feat.copy()
    roi  = feat[mask]
    rng  = roi.max() - roi.min()
    feat[mask] = (roi - roi.min()) / (rng + 1e-9)
    return feat

def _sliding(func, img, mask, win, desc=""):
    pad    = win // 2
    padded = np.pad(img.astype(float), pad, mode="reflect")
    feat   = np.zeros(img.shape, dtype=float)
    ys, xs = np.where(mask)
    if len(ys) == 0:
        return feat
    t0 = time.time()
    for i, (r, c) in enumerate(zip(ys, xs)):
        feat[r, c] = func(padded[r:r + win, c:c + win])
        if (i + 1) % 1000 == 0:
            pct     = (i + 1) / len(ys) * 100
            elapsed = time.time() - t0
            eta     = elapsed / ((i + 1) / len(ys)) - elapsed
            print(f"    {desc:14s} {pct:5.1f}%  ETA {eta:.0f}s", end="\r")
    print()
    return _norm(feat, mask)


# ══════════════════════════════════════════════════════════════════════════════
#  ① GLCM Series (6 features)
#
#  * Core fix: graycomatrix returns shape (lv, lv, n_dist, n_angle)
#     correct indexing: g[:, :, 0, a] (single distance, per angle)
#     then average over all angles to obtain (lv, lv) for broadcasting
# ══════════════════════════════════════════════════════════════════════════════
GLCM_ANGLES = [0, np.pi / 4, np.pi / 2, 3 * np.pi / 4]
GLCM_LV     = 16
_gi, _gj    = np.mgrid[0:GLCM_LV, 0:GLCM_LV]   # (16,16) index matrix, precomputed

def _glcm(patch):
    """Return (lv, lv) after averaging over 4 directions"""
    q = _q(patch, GLCM_LV).astype(np.uint8)
    # shape: (lv, lv, 1, 4) -> average over last axis -> (lv, lv)
    g = graycomatrix(q, [1], GLCM_ANGLES,
                     levels=GLCM_LV, symmetric=True, normed=True)
    return g[:, :, 0, :].mean(axis=-1)   # ★ key: shape (lv, lv)

def glcm_contrast(patch):
    g = _glcm(patch)
    return float(((_gi - _gj) ** 2 * g).sum())

def glcm_correlation(patch):
    g    = _glcm(patch)
    mu_i = (_gi * g).sum()
    mu_j = (_gj * g).sum()
    si   = np.sqrt(((_gi - mu_i) ** 2 * g).sum() + 1e-9)
    sj   = np.sqrt(((_gj - mu_j) ** 2 * g).sum() + 1e-9)
    cor  = ((_gi - mu_i) * (_gj - mu_j) * g).sum() / (si * sj)
    return float(np.clip(cor, -1, 1))

def glcm_energy(patch):
    return float((_glcm(patch) ** 2).sum())

def glcm_homogeneity(patch):
    g = _glcm(patch)
    return float((g / (1 + np.abs(_gi - _gj))).sum())

def glcm_entropy(patch):
    g = _glcm(patch) + 1e-12
    return float(-(g * np.log2(g)).sum())

def glcm_maxprob(patch):
    return float(_glcm(patch).max())


# ══════════════════════════════════════════════════════════════════════════════
#  ② GLRLM Series (4 features)
# ══════════════════════════════════════════════════════════════════════════════
def _runs_1d(row):
    runs = []; j = 0
    while j < len(row):
        v = int(row[j]); rl = 1
        while j + rl < len(row) and row[j + rl] == v: rl += 1
        runs.append((v, rl)); j += rl
    return runs

def _glrlm_features(patch):
    q     = _q(patch, 16)
    sre = lre = hglre = total = 0.0
    rln_den = np.zeros(16)
    for row in q:
        for gl, rl in _runs_1d(row):
            sre        += 1.0 / (rl ** 2)
            lre        += float(rl ** 2)
            hglre      += float(gl ** 2)
            rln_den[gl] += 1.0
            total      += 1.0
    rln_num = float((rln_den ** 2).sum())
    den = total + 1e-9
    return sre / den, lre / den, rln_num / den, hglre / den

def glrlm_sre(patch):   return _glrlm_features(patch)[0]
def glrlm_lre(patch):   return _glrlm_features(patch)[1]
def glrlm_rln(patch):   return _glrlm_features(patch)[2]
def glrlm_hglre(patch): return _glrlm_features(patch)[3]


# ══════════════════════════════════════════════════════════════════════════════
#  ③ GLSZM Series (4 features)
# ══════════════════════════════════════════════════════════════════════════════
def _glszm_features(patch):
    lv  = 16
    q   = _q(patch, lv)
    sae = lae = total = 0.0
    gl_sum = np.zeros(lv)
    for gl in range(lv):
        labeled, n = nd_label(q == gl)
        for obj in range(1, n + 1):
            sz          = float((labeled == obj).sum())
            sae        += 1.0 / (sz ** 2)
            lae        += sz ** 2
            gl_sum[gl] += sz
            total      += 1.0
    gln_num = float((gl_sum ** 2).sum())
    zp_num  = total / (q.size + 1e-9)
    den     = total + 1e-9
    return gln_num / den, sae / den, lae / den, zp_num

def glszm_gln(patch): return _glszm_features(patch)[0]
def glszm_sae(patch): return _glszm_features(patch)[1]
def glszm_lae(patch): return _glszm_features(patch)[2]
def glszm_zp(patch):  return _glszm_features(patch)[3]


# ══════════════════════════════════════════════════════════════════════════════
#  ④ First-order statistics (5 features) — vectorized
# ══════════════════════════════════════════════════════════════════════════════
def compute_firstorder_maps(img, mask, win=WIN):
    f   = img.astype(np.float64)
    mu  = uniform_filter(f,       size=win)
    mu2 = uniform_filter(f ** 2,  size=win)
    mu3 = uniform_filter(f ** 3,  size=win)
    mu4 = uniform_filter(f ** 4,  size=win)

    var  = np.abs(mu2 - mu ** 2)
    std  = np.sqrt(var + 1e-9)
    skew = (mu3 - 3 * mu * mu2 + 2 * mu ** 3) / (std ** 3 + 1e-9)
    kurt = (mu4 - 4 * mu * mu3 + 6 * (mu ** 2) * mu2 - 3 * mu ** 4) \
           / (std ** 4 + 1e-9) - 3

    # Local entropy: quantized to 16 levels then per-level mean filter
    H = np.zeros_like(f)
    for lv in range(16):
        indicator = ((img >> 4) == lv).astype(np.float64)
        p = uniform_filter(indicator, size=win).clip(1e-12, 1.0)
        H -= p * np.log2(p)

    return (_norm(mu,   mask), _norm(std,  mask),
            _norm(skew, mask), _norm(kurt, mask),
            _norm(H,    mask))


# ══════════════════════════════════════════════════════════════════════════════
#  ⑤ Filter-based features (5 features) — vectorized
# ══════════════════════════════════════════════════════════════════════════════
def compute_filter_maps(img, mask):
    f = img.astype(np.float64)

    # Gabor: maximum response over 4 orientations
    gabor_resp = np.zeros_like(f)
    for theta in [0, np.pi / 4, np.pi / 2, 3 * np.pi / 4]:
        try:
            resp = _gabor_call(f.astype(np.float32),
                               frequency=0.1, theta=theta,
                               sigma_x=3.0, sigma_y=3.0)
            gabor_resp = np.maximum(gabor_resp, resp)
        except Exception as e:
            print(f"  [WARN] Gabor θ={theta:.2f} failed: {e}")

    # LBP
    lbp_img  = local_binary_pattern(img, P=8, R=1, method="uniform")
    lbp_norm = lbp_img.astype(float) / float(lbp_img.max() + 1e-9)

    # LoG
    log_img = np.abs(laplace(sk_gaussian(f, sigma=2)))

    # Gradient magnitude
    grad_img = np.abs(sobel(f))

    # Wavelet energy
    if HAS_PYWT:
        try:
            _, (cH, cV, cD) = pywt.dwt2(img.astype(float), 'db4')
            wav_energy = np.abs(cH) + np.abs(cV) + np.abs(cD)
            scale_r = int(np.ceil(img.shape[0] / wav_energy.shape[0]))
            scale_c = int(np.ceil(img.shape[1] / wav_energy.shape[1]))
            wav_up  = np.kron(wav_energy, np.ones((scale_r, scale_c), dtype=float))
            wav_up  = wav_up[:img.shape[0], :img.shape[1]]
        except Exception as e:
            print(f"  [WARN] Wavelet failed ({e}), substituting with zero map")
            wav_up = np.zeros_like(f)
    else:
        wav_up = np.zeros_like(f)

    return (_norm(gabor_resp, mask), _norm(lbp_norm, mask),
            _norm(log_img,    mask), _norm(grad_img, mask),
            _norm(wav_up,     mask))


# ══════════════════════════════════════════════════════════════════════════════
#  Compute all features
# ══════════════════════════════════════════════════════════════════════════════
print("\n  Computing GLCM series (6 features) …")
feat_contrast     = _sliding(glcm_contrast,    img, mask, WIN, "Contrast")
feat_correlation  = _sliding(glcm_correlation, img, mask, WIN, "Correlation")
feat_energy       = _sliding(glcm_energy,      img, mask, WIN, "Energy")
feat_homogeneity  = _sliding(glcm_homogeneity, img, mask, WIN, "Homogeneity")
feat_entropy_glcm = _sliding(glcm_entropy,     img, mask, WIN, "GL-Entropy")
feat_maxprob      = _sliding(glcm_maxprob,     img, mask, WIN, "MaxProb")

print("  Computing GLRLM series (4 features) …")
feat_sre   = _sliding(glrlm_sre,   img, mask, WIN, "SRE")
feat_lre   = _sliding(glrlm_lre,   img, mask, WIN, "LRE")
feat_rln   = _sliding(glrlm_rln,   img, mask, WIN, "RLN")
feat_hglre = _sliding(glrlm_hglre, img, mask, WIN, "HGLRE")

print("  Computing GLSZM series (4 features) …")
feat_gln = _sliding(glszm_gln, img, mask, WIN, "GLN")
feat_sae = _sliding(glszm_sae, img, mask, WIN, "SAE")
feat_lae = _sliding(glszm_lae, img, mask, WIN, "LAE")
feat_zp  = _sliding(glszm_zp,  img, mask, WIN, "ZP")

print("  Computing first-order features (5 features) …")
feat_mean, feat_std, feat_skew, feat_kurt, feat_loc_ent = \
    compute_firstorder_maps(img, mask, WIN)

print("  Computing filter features (5 features) …")
feat_gabor, feat_lbp, feat_log, feat_grad, feat_wav = \
    compute_filter_maps(img, mask)

print("  All features computed ✓\n")


# ══════════════════════════════════════════════════════════════════════════════
#  K-means multi-feature habitat segmentation
# ══════════════════════════════════════════════════════════════════════════════
ALL_FEATS = [
    feat_contrast, feat_correlation, feat_energy, feat_homogeneity,
    feat_entropy_glcm, feat_maxprob,
    feat_sre, feat_lre, feat_rln, feat_hglre,
    feat_gln, feat_sae, feat_lae, feat_zp,
    feat_mean, feat_std, feat_skew, feat_kurt, feat_loc_ent,
    feat_gabor, feat_lbp, feat_log, feat_grad, feat_wav,
]

print(f"  K-means habitat segmentation (K={K}, {len(ALL_FEATS)} features) …", end="", flush=True)
X = np.column_stack([f[mask] for f in ALL_FEATS])
valid_cols = np.all(np.isfinite(X), axis=0)
if not valid_cols.all():
    print(f"\n  [WARN] {(~valid_cols).sum()} columns contain NaN/Inf, removed")
    X = X[:, valid_cols]
X = StandardScaler().fit_transform(X)
km   = KMeans(n_clusters=K, random_state=42, n_init=20, max_iter=500)
flat = km.fit_predict(X)
means_order = [feat_mean[mask][flat == c].mean() for c in range(K)]
remap = {o: i + 1 for i, o in enumerate(np.argsort(means_order))}
flat  = np.array([remap[l] for l in flat])
label_map = np.zeros(mask.shape, dtype=np.int32)
label_map[mask] = flat
print(" done ✓")
for c in range(1, K + 1):
    n   = (label_map == c).sum()
    pct = n / mask.sum() * 100
    print(f"    Habitat {c}: {n:5d} px  ({pct:.1f}%)")


# ══════════════════════════════════════════════════════════════════════════════
#  Plotting helpers
# ══════════════════════════════════════════════════════════════════════════════
def _overlay(gray, feat, mask, cmap, alpha=0.75):
    bg  = np.stack([gray] * 3, axis=-1).astype(float) / 255.0
    col = cmap(feat)[:, :, :3]
    a   = mask.astype(float) * alpha
    return (bg * (1 - a[..., None]) + col * a[..., None]).clip(0, 1)

def _add_cb(ax, cmap, label):
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(0, 1))
    sm.set_array([])
    cb = plt.colorbar(sm, ax=ax, fraction=0.034, pad=0.015, shrink=0.85)
    cb.ax.tick_params(labelsize=6, colors="#cccccc")
    cb.set_label(label, fontsize=7.5, color="#cccccc", labelpad=3)
    cb.outline.set_edgecolor("#555555")

def _frame(ax, title, letter):
    #ax.text(0.02, 0.97, letter, transform=ax.transAxes, va="top", **LTR_KW)
    ax.set_title(title, fontsize=9, color="#dddddd", fontweight="bold", pad=4)
    ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values(): sp.set_edgecolor("#555555")

def _draw_feat(ax, feat, cmap_key, title, letter):
    cmap = CMAPS[cmap_key]
    ax.imshow(_overlay(img, feat, mask, cmap), interpolation="bilinear")
    _add_cb(ax, cmap, "norm.")
    ax.contour(mask.astype(np.uint8), levels=[0.5],
               colors=["#ffffff66"], linewidths=[0.8])
    _frame(ax, title, letter)

def _draw_habitat(ax, letter):
    bg   = np.stack([img] * 3, axis=-1).astype(float) / 255.0
    comp = bg * 0.28
    hab_colors = CMAPS["habitat"]
    for ci in range(K):
        hx_clean = hab_colors[ci].lstrip("#")
        rgb = np.array([int(hx_clean[i:i+2], 16) / 255.0 for i in (0, 2, 4)])
        zone = label_map == (ci + 1)
        for ch in range(3):
            comp[zone, ch] = bg[zone, ch] * 0.28 + rgb[ch] * 0.72
    ax.imshow(comp.clip(0, 1), interpolation="bilinear")
    for ci in range(K):
        hx   = hab_colors[ci]
        zone = (label_map == (ci + 1)).astype(np.uint8)
        ax.contour(zone, levels=[0.5], colors=[hx], linewidths=[1.5], alpha=0.95)
    ax.contour(mask.astype(np.uint8), levels=[0.5],
               colors=["#ffffff"], linewidths=[1.8], alpha=0.75)
    handles = [Patch(facecolor=hab_colors[i], edgecolor="white",
                     linewidth=0.7, label=f"Habitat {i+1}")
               for i in range(K)]
    leg = ax.legend(handles=handles, loc="lower right", fontsize=8,
                    framealpha=0.55, edgecolor="#777777", facecolor="#1a1a2ecc")
    for t in leg.get_texts(): t.set_color("white")
    
    # Set title without extra frame letter
    ax.set_title(f"K-means Habitat (K={K})", fontsize=9, color="#dddddd", fontweight="bold", pad=4)
    ax.set_xticks([])
    ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_edgecolor("#555555")


# ══════════════════════════════════════════════════════════════════════════════
#  Plot full-spectrum heatmap (5 rows × 5 columns) + habitat
# ══════════════════════════════════════════════════════════════════════════════
PANEL_DEFS = [
    # (feat, cmap_key, title, letter)
    (feat_contrast,    "contrast",    "GLCM Contrast",        "1"),
    (feat_correlation, "correlation", "GLCM Correlation",     "2"),
    (feat_energy,      "energy",      "GLCM Energy",          "3"),
    (feat_homogeneity, "homogeneity", "GLCM Homogeneity",     "4"),
    (feat_entropy_glcm,"entropy",     "GLCM Entropy",         "5"),
    (feat_maxprob,     "maxprob",     "GLCM MaxProb",         "6"),
    (feat_sre,         "sre",         "GLRLM SRE",            "7"),
    (feat_lre,         "lre",         "GLRLM LRE",            "8"),
    (feat_rln,         "rln",         "GLRLM RLN",            "9"),
    (feat_hglre,       "hglre",       "GLRLM HGLRE",          "10"),
    (feat_gln,         "gln",         "GLSZM GLN",            "11"),
    (feat_sae,         "sae",         "GLSZM SAE",            "12"),
    (feat_lae,         "lae",         "GLSZM LAE",            "13"),
    (feat_zp,          "zp",          "GLSZM ZP",             "14"),
    (feat_mean,        "mean",        "1st Mean",             "15"),
    (feat_std,         "std",         "1st Std",              "16"),
    (feat_skew,        "skewness",    "1st Skewness",         "17"),
    (feat_kurt,        "kurtosis",    "1st Kurtosis",         "18"),
    (feat_loc_ent,     "loc_entropy", "1st Local Entropy",    "19"),
    (feat_gabor,       "gabor",       "Filter Gabor",         "20"),
    (feat_lbp,         "lbp",         "Filter LBP",           "21"),
    (feat_log,         "log",         "Filter LoG",           "22"),
    (feat_grad,        "gradient",    "Filter Gradient",      "23"),
    (feat_wav,         "wavelet",     "Filter Wavelet",       "24"),
]

N_PANELS = len(PANEL_DEFS) + 1   # 24 feature maps + 1 habitat = 25
COLS     = 5
ROWS     = int(np.ceil(N_PANELS / COLS))

print(f"\n  Rendering {N_PANELS} subplots ({ROWS}×{COLS}) …")
fig = plt.figure(figsize=(COLS * 4.4, ROWS * 4.6), facecolor=BG)
gs  = gridspec.GridSpec(ROWS, COLS, figure=fig,
                        wspace=0.06, hspace=0.30,
                        left=0.01, right=0.99,
                        top=0.96, bottom=0.02)

for idx, (feat, cmap_key, title, letter) in enumerate(PANEL_DEFS):
    r, c = divmod(idx, COLS)
    ax   = fig.add_subplot(gs[r, c])
    ax.set_facecolor(BG)
    _draw_feat(ax, feat, cmap_key, title, letter)

# Last cell: habitat segmentation
r_last, c_last = divmod(N_PANELS - 1, COLS)
ax_hab = fig.add_subplot(gs[r_last, c_last])
ax_hab.set_facecolor(BG)
_draw_habitat(ax_hab, "H")

fig.suptitle(
    f"Radiomics Full-Spectrum Heatmaps  ·  {patient_id}  ({label_class})\n"
    f"GLCM(6) · GLRLM(4) · GLSZM(4) · First-Order(5) · Filter(5) · K-means Habitat",
    fontsize=12, fontweight="bold", color="#eeeeee", y=0.995
)

main_path = os.path.join(OUT_DIR, f"heatmaps_all_{patient_id}.png")
fig.savefig(main_path, dpi=200, bbox_inches="tight", facecolor=BG)
print(f"  ✓ Full-spectrum heatmap: {main_path}")
plt.close(fig)


# ══════════════════════════════════════════════════════════════════════════════
#  Group saving (each group in a separate figure)
# ══════════════════════════════════════════════════════════════════════════════
GROUPS = {
    "GLCM"      : PANEL_DEFS[0:6],
    "GLRLM"     : PANEL_DEFS[6:10],
    "GLSZM"     : PANEL_DEFS[10:14],
    "FirstOrder": PANEL_DEFS[14:19],
    "Filter"    : PANEL_DEFS[19:24],
}

for grp_name, panels in GROUPS.items():
    n  = len(panels)
    gc = min(n, 4)
    gr = int(np.ceil(n / gc))
    f2 = plt.figure(figsize=(gc * 4.8, gr * 4.8), facecolor=BG)
    g2 = gridspec.GridSpec(gr, gc, figure=f2,
                           wspace=0.07, hspace=0.30,
                           left=0.01, right=0.99,
                           top=0.90, bottom=0.03)
    for idx2, (feat, cmap_key, title, letter) in enumerate(panels):
        r2, c2 = divmod(idx2, gc)
        ax2    = f2.add_subplot(g2[r2, c2])
        ax2.set_facecolor(BG)
        _draw_feat(ax2, feat, cmap_key, title, letter)
    f2.suptitle(f"{grp_name} Features  ·  {patient_id}",
                fontsize=12, fontweight="bold", color="#eeeeee")
    p2 = os.path.join(OUT_DIR, f"heatmaps_{grp_name}_{patient_id}.png")
    f2.savefig(p2, dpi=250, bbox_inches="tight", facecolor=BG)
    print(f"  ✓ {grp_name}: {p2}")
    plt.close(f2)

# Habitat separately
f3, a3 = plt.subplots(figsize=(6, 6), facecolor=BG)
a3.set_facecolor(BG)
_draw_habitat(a3, "H")
f3.tight_layout(pad=0.4)
p3 = os.path.join(OUT_DIR, f"heatmaps_Habitat_{patient_id}.png")
f3.savefig(p3, dpi=250, bbox_inches="tight", facecolor=BG)

plt.close(f3)


# ══════════════════════════════════════════════════════════════════════════════
#  Summary
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 60)

print(f"  Patient : {patient_id}  ({label_class})")
print(f"  Output : {os.path.abspath(OUT_DIR)}")
print("  Files :")
for f in sorted(os.listdir(OUT_DIR)):
    kb = os.path.getsize(os.path.join(OUT_DIR, f)) // 1024
    print(f"    {f:<52}  {kb:>5} KB")
print("=" * 60)